# L20 — Routing Logic: Probabilistic, Attribute-Based, and State-Dependent

**Module**: M06 | **Chapter**: 8 | **Lecture**: L20

## Learning Objectives
By the end of this notebook you will be able to:
1. Implement probabilistic branching (fixed routing fractions) in SimPy.
2. Route entities based on their attributes (condition-based routing).
3. Implement state-dependent routing (shortest-queue policy).
4. Model feedback loops (rework) and their effect on throughput.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**
---

In [ ]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Callable

## 1. Probabilistic Routing

After a service stage, entities are routed to the next stage with fixed probabilities.  
Example: 70% of patients go to the exam room; 30% are discharged after triage.

In [ ]:
def probabilistic_routing_sim(lam: float,
                               mu_stage1: float,
                               mu_stage2: float,
                               p_continue: float,    # prob of going to stage 2
                               c1: int = 1, c2: int = 2,
                               sim_time: float = 100_000,
                               seed: int = 0) -> dict:
    """
    Two-stage system with probabilistic routing after stage 1.
    Fraction p_continue of patients proceed to stage 2.
    """
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    s1 = simpy.Resource(env, capacity=c1)
    s2 = simpy.Resource(env, capacity=c2)

    s1_waits, s2_waits, total_sojourns = [], [], []

    def patient():
        arrival = env.now

        # Stage 1
        with s1.request() as req:
            yield req
            w1 = env.now - arrival
            yield env.timeout(rng.exponential(1.0 / mu_stage1))
        s1_waits.append(w1)

        # Route
        if rng.random() < p_continue:
            t_s2_arrive = env.now
            with s2.request() as req:
                yield req
                w2 = env.now - t_s2_arrive
                yield env.timeout(rng.exponential(1.0 / mu_stage2))
            s2_waits.append(w2)

        total_sojourns.append(env.now - arrival)

    def arrivals():
        while True:
            yield env.timeout(rng.exponential(1.0 / lam))
            env.process(patient())

    env.process(arrivals())
    env.run(until=sim_time)

    return {
        'Wq_s1': np.mean(s1_waits),
        'Wq_s2': np.mean(s2_waits) if s2_waits else 0.0,
        'W_total': np.mean(total_sojourns),
        'n_served_s1': len(s1_waits),
        'n_served_s2': len(s2_waits),
        'actual_p': len(s2_waits) / len(s1_waits) if s1_waits else 0,
    }


# Sensitivity: vary the routing fraction
lam_r = 5.0
print(f"λ={lam_r}, μ₁=20, μ₂=4 (c₁=1, c₂=2)")
print(f"{'p_continue':>12s}  {'Wq_s2 (min)':>12s}  {'W_total (min)':>14s}  {'actual_p':>10s}")
print('-' * 55)
for p in [0.3, 0.5, 0.7, 0.9]:
    r = probabilistic_routing_sim(lam_r, 20.0, 4.0, p, c1=1, c2=2, seed=0)
    print(f"{p:>12.1f}  {r['Wq_s2']*60:>12.2f}  {r['W_total']*60:>14.2f}  {r['actual_p']:>10.3f}")

## 2. Attribute-Based Routing

Routing decisions depend on entity attributes (acuity level, customer type, job category).

In [ ]:
def attribute_routing_sim(lam: float, mu_general: float, mu_specialist: float,
                           p_complex: float,    # fraction needing specialist
                           c_gen: int = 2, c_spec: int = 1,
                           sim_time: float = 100_000, seed: int = 0) -> dict:
    """
    Triage system:
    - p_complex: arrive and go directly to specialist queue
    - 1-p_complex: arrive and go to general queue
    No shared first stage — routing at arrival.
    """
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    general    = simpy.Resource(env, capacity=c_gen)
    specialist = simpy.Resource(env, capacity=c_spec)

    records = []

    def patient(is_complex: bool):
        arrival = env.now
        res = specialist if is_complex else general
        mu  = mu_specialist if is_complex else mu_general
        with res.request() as req:
            yield req
            wait = env.now - arrival
            yield env.timeout(rng.exponential(1.0 / mu))
        records.append({'complex': is_complex, 'wait': wait,
                         'sojourn': env.now - arrival})

    def arrivals():
        while True:
            yield env.timeout(rng.exponential(1.0 / lam))
            is_complex = rng.random() < p_complex
            env.process(patient(is_complex))

    env.process(arrivals())
    env.run(until=sim_time)

    df = pd.DataFrame(records)
    return df


lam_a = 8.0
df_attr = attribute_routing_sim(lam_a, mu_general=6.0, mu_specialist=2.0,
                                 p_complex=0.3, c_gen=2, c_spec=1, seed=42)

print("Attribute-based routing (λ=8, μ_gen=6, μ_spec=2, c_gen=2, c_spec=1):")
for is_c, label in [(False, 'General'), (True, 'Specialist')]:
    sub = df_attr[df_attr['complex'] == is_c]
    print(f"  {label:12s}: n={len(sub):5,}  Wq={sub['wait'].mean()*60:.2f} min  "
          f"W={sub['sojourn'].mean()*60:.2f} min")

## 3. State-Dependent Routing: Shortest Queue

Customers choose the server with the shortest current queue (join-the-shortest-queue, JSQ).  
This is a **state-dependent** decision — it reads the current system state.

In [ ]:
def jsq_simulation(lam: float, mu: float, c: int,
                   sim_time: float = 100_000, seed: int = 0) -> dict:
    """
    Join-the-shortest-queue (JSQ) with c dedicated servers.
    Each server has its own queue; new customers join the shortest.
    Compare to single pooled queue (M/M/c).
    """
    rng = np.random.default_rng(seed)
    env = simpy.Environment()

    # c separate single-server queues
    servers = [simpy.Resource(env, capacity=1) for _ in range(c)]
    waits = []

    def customer():
        arrival = env.now
        # JSQ: pick server with fewest waiting+in-service
        chosen = min(servers, key=lambda s: s.count + len(s.queue))
        with chosen.request() as req:
            yield req
            wait = env.now - arrival
            yield env.timeout(rng.exponential(1.0 / mu))
        waits.append(wait)

    def arrivals():
        while True:
            yield env.timeout(rng.exponential(1.0 / lam))
            env.process(customer())

    env.process(arrivals())
    env.run(until=sim_time)
    return {'Wq': np.mean(waits), 'n_served': len(waits)}


from math import factorial

def erlang_c_wq(c, lam, mu):
    rho = lam / (c * mu)
    if rho >= 1:
        return float('inf')
    a = lam / mu
    s = sum(a**n / factorial(n) for n in range(c))
    last = (a**c / factorial(c)) / (1 - rho)
    C = last / (s + last)
    return C / (c * mu - lam)


lam_j, mu_j = 8.0, 5.0   # ρ_per_server = lam/(c*mu)
for c in [2, 3, 4]:
    jsq = jsq_simulation(lam_j, mu_j, c, seed=0)
    mmc_wq = erlang_c_wq(c, lam_j, mu_j)
    ded_wq = erlang_c_wq(1, lam_j/c, mu_j)  # dedicated single-server each
    print(f"c={c}:  JSQ Wq={jsq['Wq']*60:.2f} min  |  "
          f"Pooled M/M/{c}: {mmc_wq*60:.2f} min  |  "
          f"Dedicated M/M/1: {ded_wq*60:.2f} min")

## 4. Feedback Loop: Rework

A fraction of jobs fail inspection and return to the queue for rework.  
This creates a feedback loop that increases effective load.

In [ ]:
def rework_sim(lam: float, mu_process: float, mu_inspect: float,
               p_rework: float,   # probability of failing inspection
               sim_time: float = 100_000, seed: int = 0) -> dict:
    """
    Two-stage system: Process → Inspect.
    p_rework fraction of jobs fail inspection and rejoin the Process queue.
    """
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    processor = simpy.Resource(env, capacity=1)
    inspector = simpy.Resource(env, capacity=1)

    # Effective arrival rate = lam / (1 - p_rework)  (geometric series)
    records = []

    def job(first_pass: bool = True):
        arrival_this_pass = env.now

        with processor.request() as req:
            yield req
            wq_proc = env.now - arrival_this_pass
            yield env.timeout(rng.exponential(1.0 / mu_process))

        with inspector.request() as req:
            yield req
            yield env.timeout(rng.exponential(1.0 / mu_inspect))

        if rng.random() < p_rework:
            env.process(job(first_pass=False))  # rework
        else:
            records.append({'first_pass': first_pass})

    def arrivals():
        while True:
            yield env.timeout(rng.exponential(1.0 / lam))
            env.process(job())

    env.process(arrivals())
    env.run(until=sim_time)

    n_rework = sum(1 for r in records if not r['first_pass'])
    return {
        'throughput': len(records) / sim_time,
        'pct_reworked': n_rework / len(records) * 100 if records else 0,
        'effective_lam': lam / (1 - p_rework),  # theoretical
    }


lam_rw, mu_p, mu_i = 3.0, 5.0, 8.0
print(f"λ={lam_rw}, μ_process={mu_p}, μ_inspect={mu_i}")
print(f"{'p_rework':>10s}  {'effective_λ':>12s}  {'ρ_processor':>12s}  {'throughput':>12s}")
print('-' * 55)
for p in [0.0, 0.10, 0.20, 0.30]:
    r = rework_sim(lam_rw, mu_p, mu_i, p, seed=0)
    eff_lam = lam_rw / (1 - p) if p < 1 else float('inf')
    rho_eff = eff_lam / mu_p
    print(f"{p:>10.2f}  {eff_lam:>12.3f}  {rho_eff:>12.3f}  {r['throughput']:>12.4f}")

print()
print("At p_rework=0.30, effective load on processor = 3/(1-0.30)/5 = 0.857")
print("The system approaches instability even though the nominal ρ = 3/5 = 0.60.")

---
## Try It Yourself

1. **Overflow routing**: Implement a two-server system where one server is preferred (first choice). If the preferred server has > K customers in queue, the customer routes to the secondary server instead. Compare overflow routing to JSQ and M/M/2 for K ∈ {2, 5, 10}.

2. **Critical path with feedback**: A manufacturing cell has 3 operations in series. After operation 3, 15% of items fail inspection and loop back to operation 1. Simulate 30 replications and compute the average throughput time (entry at operation 1 to final inspection pass). Verify that effective load on each station equals λ/(1−p_rework).

3. **Adaptive routing**: Implement a routing rule that switches between JSQ and random assignment based on queue imbalance: if the longest queue is > 2× the shortest, use JSQ; otherwise route randomly. Compare this adaptive rule to always-JSQ and always-random.